# Threads vs Processes

Same workload, two pools, IO vs CPU. Run inside `if __name__ == '__main__'`-safe blocks for spawn mode.

In [ ]:
import time, math, os
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def io_task(_=None):
    time.sleep(0.1)

def cpu_task(_=None):
    s = 0
    for i in range(2, 80_000):
        if all(i % p for p in range(2, int(math.isqrt(i))+1)):
            s += i
    return s

def bench(label, Pool, fn, n, workers):
    t = time.perf_counter()
    with Pool(max_workers=workers) as ex:
        list(ex.map(fn, range(n)))
    print(f'{label:18} workers={workers:>2}  ms={(time.perf_counter()-t)*1000:8.1f}')

n = os.cpu_count() or 4
print('--- IO bound ---')
for w in (1, 4, 8):
    bench('thread/IO',    ThreadPoolExecutor,  io_task,  20, w)
    bench('process/IO',   ProcessPoolExecutor, io_task,  20, w)
print('--- CPU bound ---')
for w in (1, 2, n):
    bench('thread/CPU',   ThreadPoolExecutor,  cpu_task, n,  w)
    bench('process/CPU',  ProcessPoolExecutor, cpu_task, n,  w)

## Reflect

- Why are threads roughly equivalent to processes for IO?
- Why do threads barely speed up CPU work?
- Why do processes win CPU but lose to threads for short IO tasks?